# Day 10: Experiment Tracking & MLOps Basics

## Why Track Experiments?
Once you've trained 20 versions of a model with different hyperparameters, 'which run gave 0.91 F1?' becomes impossible to answer from memory alone.

In [ ]:
# pip install mlflow

## Logging a Run with MLflow
mlflow.log_param/log_metric/sklearn.log_model record everything about one training run so it can be compared and reproduced later.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run():
    n_estimators = 100
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)
    f1 = f1_score(y_test, model.predict(X_test), average="macro")

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_metric("f1_macro", f1)
    mlflow.sklearn.log_model(model, "model")

print("Logged run with F1:", f1)

## Comparing Runs
The MLflow UI shows every run side by side, sortable by any logged metric - no more spreadsheet of results.

In [ ]:
# terminal:
# mlflow ui
# then open http://localhost:5000 to compare all runs visually

## Model Registry Concept
A registry promotes a specific run's model through stages (Staging -> Production) so the serving API always knows exactly which model is live.

In [ ]:
# after finding the best run in the UI:
# mlflow.register_model("runs:/<run_id>/model", "iris-classifier")
# then transition it: None -> Staging -> Production, tracked centrally
# your FastAPI service (Day 9) loads "models:/iris-classifier/Production" instead of a local .pkl

## CI/CD for ML — the Idea
Every push automatically re-runs tests (Day 7) and, optionally, retrains/evaluates the model before it's allowed to deploy.

In [ ]:
# .github/workflows/ci.yml  (GitHub Actions)
# name: CI
# on: [push]
# jobs:
#   test:
#     runs-on: ubuntu-latest
#     steps:
#       - uses: actions/checkout@v4
#       - uses: actions/setup-python@v5
#         with: { python-version: "3.11" }
#       - run: pip install -r requirements.txt
#       - run: pytest